# Chennai Job Market Insights — Business Analyst / Data Analyst

## Portfolio case study
This notebook follows a simple Business Analyst workflow: **business question → data check → KPI → comparison → recommendation**.

> **Data transparency:** the 120 records are synthetic/illustrative. They are not live scraped labour-market data.

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('sample_listings_chennai.csv')
df['Salary Midpoint LPA'] = (df['Min Salary LPA'] + df['Max Salary LPA']) / 2
df['Salary Spread LPA'] = df['Max Salary LPA'] - df['Min Salary LPA']
df['Listing ID'] = range(1, len(df) + 1)
display(df.head())


## 1. Can we trust the dataset structure?


In [ ]:
quality = {
    'Rows': len(df),
    'Columns': len(df.columns),
    'Duplicate rows': int(df.duplicated().sum()),
    'Unique companies': int(df['Company'].nunique()),
    'Unique Chennai areas': int(df['Location'].nunique()),
    'Blank skills': int(df['Skills'].isna().sum()),
}
pd.Series(quality)


## 2. Executive questions
What roles, salaries and skills dominate the sample?


In [ ]:
skills_per_listing = (
    df.assign(Skill=df['Skills'].str.split(';'))
      .explode('Skill')
      .assign(Skill=lambda x: x['Skill'].str.strip())
      [['Listing ID','Skill']]
      .drop_duplicates()
)
kpis = pd.Series({
    'Sample listings': len(df),
    'Median salary midpoint (LPA)': round(df['Salary Midpoint LPA'].median(), 2),
    'Most common role': df['Job Title'].value_counts().idxmax(),
    'Most requested skill': skills_per_listing['Skill'].value_counts().idxmax(),
})
kpis


## 3. Salary: what is the typical signal by role?


In [ ]:
salary_by_role = (
    df.groupby('Job Title')
      .agg(Listings=('Job Title','size'), Median_Salary_Midpoint_LPA=('Salary Midpoint LPA','median'))
      .sort_values('Median_Salary_Midpoint_LPA', ascending=False)
)
display(salary_by_role.round(2))


## 4. Skills: which capabilities appear most often?
Each listing counts once per skill, even if the text repeats the same skill.


In [ ]:
skill_demand = skills_per_listing.groupby('Skill').size().rename('Listings').to_frame()
skill_demand['Share_of_listings_pct'] = (skill_demand['Listings'] / len(df) * 100).round(1)
display(skill_demand.sort_values('Listings', ascending=False).head(15))


## 5. Where are the opportunities represented?


In [ ]:
for col, label in [('Location','Chennai area'),('Industry','Industry'),('Work Arrangement','Work style'),('Job Title','Role')]:
    out = df[col].value_counts().rename_axis(label).reset_index(name='Listings')
    out['Share_pct'] = (out['Listings'] / len(df) * 100).round(1)
    print(f'\n{label}')
    display(out)


## 6. Business Analyst vs Data Analyst
Compare the percentage of listings within each role that mention each skill.


In [ ]:
ba_da = df[df['Job Title'].isin(['Business Analyst','Data Analyst'])][['Listing ID','Job Title','Skills']].copy()
role_skills = (
    ba_da.assign(Skill=ba_da['Skills'].str.split(';'))
        .explode('Skill')
        .assign(Skill=lambda x: x['Skill'].str.strip())
        [['Listing ID','Job Title','Skill']]
        .drop_duplicates()
)
role_sizes = ba_da.groupby('Job Title')['Listing ID'].nunique()
profile = role_skills.groupby(['Job Title','Skill'])['Listing ID'].nunique().rename('Listings').reset_index()
profile['Demand_pct'] = (profile['Listings'] / profile['Job Title'].map(role_sizes) * 100).round(1)
display(profile.pivot(index='Skill', columns='Job Title', values='Demand_pct').fillna(0).sort_index())


## 7. Decision support
Use the analysis to turn observations into actions. Keep conclusions labelled as **illustrative signals from this sample**.


In [ ]:
top_role = df['Job Title'].value_counts().idxmax()
top_skill = skill_demand['Listings'].idxmax()
top_area = df['Location'].value_counts().idxmax()
top_industry = df['Industry'].value_counts().idxmax()
print(f'1. Build the common baseline: {top_skill} is the most frequently mentioned skill in this sample.')
print(f'2. Role focus: {top_role} has the largest listing count in this sample.')
print(f'3. Targeting: {top_area} and {top_industry} are the most represented area and industry in this sample.')
print('4. Next step: compare the BA and DA profile before deciding which role path to prioritise.')
